# BƯỚC 1 & 2: KHẢO SÁT & KIỂM TRA CHẤT LƯỢNG DỮ LIỆU THÔ (DATA QUALITY & PROFILING)

---
**Mục tiêu của Notebook:**
1. **Trích xuất dữ liệu cần thiết (Feature Selection)** từ kho dữ liệu thô Bronze (bronze_listings + bronze_details_valid).
2. **Kiểm tra cấu trúc và kiểu dữ liệu (Schema & Data Types)** của toàn bộ các trường tuyển dụng.
3. **Đo lường giá trị khuyết (Missing / Null Report)**: Thống kê số lượng Null/None và chuỗi rỗng trên từng cột.
4. **Kiểm tra trùng lặp (Duplicates Report)**: Phát hiện trùng lặp theo ID nguồn, URL bài đăng và thực thể công việc [Công ty + Chức danh].
5. Đưa ra cơ sở thực tế để thực hiện bước Tiền xử lý & Làm sạch dữ liệu tiếp theo.

In [1]:
import os
import json
import pandas as pd
import numpy as np

from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print(' Môi trường phân tích dữ liệu đã sẵn sàng!')

 Môi trường phân tích dữ liệu đã sẵn sàng!


## 1. Nạp Dữ Liệu Master Từ Tầng Bronze

Đọc 2 tệp tích lũy chuẩn từ `data/bronze/`:
- `bronze_listings_master.json`: Bảng danh mục tin tuyển dụng (kèm tags, logo, trạng thái crawl).
- `bronze_details_valid.json`: Bảng chi tiết việc làm hợp lệ sau khi qua bộ kiểm định Data Quality.

In [2]:
# Duong dan toi 2 file Bronze Master (tuong thich moi thu muc lam viec)
def resolve_path(rel_subpath):
    candidates = [
        rel_subpath,
        f"../{rel_subpath}",
        f"../../{rel_subpath}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath(rel_subpath)

listings_path = resolve_path('data/bronze/bronze_listings_master.json')
details_path = resolve_path('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)

df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(details_data)

print(f'Dang doc Listings tu: {listings_path}')
print(f'Dang doc Details tu : {details_path}\n')
print(f'Bang Listings: {len(df_listings):,} dong, {df_listings.shape[1]} cot')
print(f'Bang Details : {len(df_details):,} dong, {df_details.shape[1]} cot\n')

# Thong ke so luong tin theo tung san
summary_sources = pd.DataFrame({
    'So Listings': df_listings['source'].value_counts(),
    'So Details Valid': df_details['source'].value_counts()
})
display(summary_sources)


Dang doc Listings tu: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
Dang doc Details tu : c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

Bang Listings: 5,524 dong, 20 cot
Bang Details : 5,491 dong, 35 cot



,So Listings,So Details Valid
source,,
topcv,3594,3561
vietnamworks,1749,1749
topdev,181,181


## 2. Ghép 2 Bảng (Join Listings + Details)

Ghép theo cặp khóa định danh duy nhất: `['source', 'source_job_id']`.
- **Mục tiêu**: Bổ sung các cột giá trị từ Listings mà Details có thể chưa có (như `company_logo_url`, `skills_raw`, `posted_at_raw`, `raw_file_name` của listing).
- Giữ nguyên toàn bộ các trường nội dung chi tiết sâu (`overview_raw`, `responsibilities_raw`, `requirements_raw`, `benefits_raw`, `work_mode_raw`, `company_size_raw`...) từ Details.

In [3]:
# Chọn các cột bổ sung từ Listings (tránh xung đột trùng lặp cột)
listing_extra_cols = ['source', 'source_job_id', 'company_logo_url', 'skills_raw', 'posted_at_raw', 'raw_file_name']
df_list_sub = df_listings[listing_extra_cols].drop_duplicates(subset=['source', 'source_job_id']).rename(
    columns={'raw_file_name': 'listing_file_name'}
)

# Ghép bảng Details với Listings theo ['source', 'source_job_id']
df_merged = pd.merge(
    df_details,
    df_list_sub,
    on=['source', 'source_job_id'],
    how='inner'
)

print(f' Kết quả ghép bảng (Join):')
print(f'  • Tổng số dòng sau khi ghép : {len(df_merged):,} dòng')
print(f'  • Tổng số cột sau khi ghép  : {df_merged.shape[1]} cột')
print(f'  • Tỷ lệ khớp khóa (Match)  : {len(df_merged)/len(df_details)*100:.2f}% (Khớp hoàn toàn 100%)')

 Kết quả ghép bảng (Join):
  • Tổng số dòng sau khi ghép : 5,491 dòng
  • Tổng số cột sau khi ghép  : 39 cột
  • Tỷ lệ khớp khóa (Match)  : 100.00% (Khớp hoàn toàn 100%)


## 3. Thống Kê Chi Tiết: Info, Missing Values (Null / Rỗng), và Duplicates (Dup)

Khảo sát toàn diện chất lượng dữ liệu của bảng sau khi ghép:
1. **Cấu trúc & Kiểu dữ liệu (`.info()`)**.
2. **Thống kê giá trị khuyết (`null` và chuỗi rỗng `""`)**: Tính số lượng và tỷ lệ % khuyết trên từng trường dữ liệu.
3. **Kiểm tra trùng lặp (`duplicates`)**: Kiểm tra trùng theo ID nguồn, theo URL, và theo cặp chức danh - công ty.

In [4]:
# 1. Hiển thị thông tin tổng quan kiểu dữ liệu
print('--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---')
df_merged.info(verbose=True, show_counts=True)

# 2. Thống kê Missing Values (Null / None và Chuỗi Rỗng "")
null_counts = df_merged.isnull().sum()
empty_counts = (df_merged == '').sum()
total_missing = null_counts + empty_counts
missing_pct = (total_missing / len(df_merged) * 100).round(2)

df_missing_report = pd.DataFrame({
    'Kiểu Dữ Liệu': df_merged.dtypes.astype(str),
    'Số lượng Null/None': null_counts,
    'Số lượng Chuỗi Rỗng ("")': empty_counts,
    'Tổng Số Khuyết': total_missing,
    'Tỷ Lệ Khuyết (%)': missing_pct
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('\n--- 2. BẢNG THỐNG KÊ GIÁ TRỊ KHUYẾT (MISSING / NULL REPORT) ---')
display(df_missing_report)

# 3. Kiểm tra Trùng lặp (Duplicates Report)
dup_source_id = df_merged.duplicated(subset=['source', 'source_job_id']).sum()
dup_job_url = df_merged.duplicated(subset=['job_url']).sum()
dup_title_company = df_merged.duplicated(subset=['title_raw', 'company_name_raw']).sum()

print('--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---')
df_dup_report = pd.DataFrame({
    'Tiêu Chí Trùng Lặp': [
        'Theo cặp khóa nguồn [source, source_job_id]',
        'Theo đường dẫn [job_url]',
        'Theo cùng vị trí & cùng công ty [title_raw, company_name_raw]'
    ],
    'Số Bản Ghi Trùng': [dup_source_id, dup_job_url, dup_title_company],
    'Tỷ Lệ Trùng (%)': [
        round(dup_source_id / len(df_merged) * 100, 2),
        round(dup_job_url / len(df_merged) * 100, 2),
        round(dup_title_company / len(df_merged) * 100, 2)
    ]
})
display(df_dup_report)

--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 5491 entries, 0 to 5490
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   detail_id             5491 non-null   str   
 1   source                5491 non-null   str   
 2   source_job_id         5491 non-null   str   
 3   job_url               5491 non-null   str   
 4   title_raw             5491 non-null   str   
 5   company_name_raw      5491 non-null   str   
 6   company_url           5491 non-null   str   
 7   company_size_raw      5491 non-null   str   
 8   company_location_raw  5491 non-null   str   
 9   industry_raw          5491 non-null   str   
 10  salary_raw            5491 non-null   str   
 11  experience_raw        5491 non-null   str   
 12  education_raw         5491 non-null   str   
 13  location_raw          5491 non-null   str   
 14  work_mode_raw         5491 non-null   str   
 15  employmen

,Kiểu Dữ Liệu,Số lượng Null/None,"Số lượng Chuỗi Rỗng ("""")",Tổng Số Khuyết,Tỷ Lệ Khuyết (%)
preferred_skills_raw,str,0,5491,5491,100.00
updated_at_raw,str,0,3742,3742,68.15
overview_raw,str,0,3415,3415,62.19
company_size_raw,str,0,1137,1137,20.71
education_raw,str,0,846,846,15.41
working_time_raw,str,0,319,319,5.81
work_mode_raw,str,0,46,46,0.84
company_logo_url,str,0,40,40,0.73
requirements_raw,str,0,27,27,0.49
required_skills_raw,str,0,5,5,0.09


--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---


,Tiêu Chí Trùng Lặp,Số Bản Ghi Trùng,Tỷ Lệ Trùng (%)
0,"Theo cặp khóa nguồn [source, source_job_id]",0,0.00
1,Theo đường dẫn [job_url],0,0.00
2,"Theo cùng vị trí & cùng công ty [title_raw, company_name...",293,5.34


## 4. Xem Mẫu Dữ Liệu Sau Khi Ghép (Toàn Bộ 39 Cột - Dễ Quan Sát)

Vì dữ liệu sau khi ghép có **39 cột** (bao gồm cả các trường text dài và metadata), nếu để một bảng phẳng nằm ngang sẽ rất khó quan sát. Chúng ta tổ chức hiển thị theo các cách sau để bạn dễ kiểm tra nhất:

1.  **Cách 1: Hiển thị Bảng Chuyển Vị (Transposed `.T`)**: Chuyển toàn bộ 39 cột thành danh sách dọc. Mỗi tin là một cột  Dễ dàng cuộn dọc đọc trọn vẹn từng trường mà không bị tràn màn hình.
2.  **Cách 2: Hiển thị 10 Dòng Theo Từng Nhóm Nghiệp Vụ**: Phân tách 39 cột thành 5 nhóm có cấu trúc (*Định danh, Công ty, Lương & Đãi ngộ, Kỹ năng, Nội dung JD, Data Quality*).
3.  **Cách 3: Bảng Phẳng Đầy Đủ (Toàn bộ 39 cột x 10 dòng)**: Mở rộng `display.max_columns = None` để bạn có thể cuộn ngang xem tất cả các cột nếu cần.

In [5]:
# --- CÁCH 1: HIỂN THỊ DẠNG CHUYỂN VỊ (TRANSPOSED VIEW) ---
# 39 cột sẽ hiển thị theo chiều dọc (Index), các mẫu tin hiển thị theo cột
print(' [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:')
sample_3 = df_merged.sample(3, random_state=42)
display(sample_3.T)


 [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:


,4562,691,3007
detail_id,topcv_2344892,topcv_2190275,topcv_2298417
source,topcv,topcv,topcv
source_job_id,2344892,2190275,2298417
job_url,https://www.topcv.vn/viec-lam/chuyen-vien-graphic-design...,https://www.topcv.vn/viec-lam/project-manager/2190275.html,https://www.topcv.vn/viec-lam/senior-design-executive/22...
title_raw,Chuyên Viên Graphic Designer (F&B),Project Manager,Senior Design Executive
company_name_raw,CÔNG TY TNHH THƯƠNG MẠI VÀ DỊCH VỤ ĐỒ UỐNG VIETBLEND,Công ty TNHH Yopaz,CÔNG TY CỔ PHẦN THƯƠNG MẠI KOVA
company_url,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-va-...,https://www.topcv.vn/cong-ty/cong-ty-tnhh-yopaz/56828.html,https://www.topcv.vn/cong-ty/kova-commerce/263712.html
company_size_raw,25-99 nhân viên,100-499 nhân viên,100-499 nhân viên
company_location_raw,"Số 10/1, ngõ 55 Huỳnh Thúc Kháng, P. Láng, Hà Nội","Tầng 3, Viet Tower, số 1 Thái Hà, Đống Đa, Hà Nội","92G-92H Nguyễn Hữu Cảnh, Phường Thạnh Mỹ Tây, TP Hồ Chí ..."
industry_raw,"Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...","IT phần mềm, Công nghệ thông tin","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I..."


In [6]:
# ---  XEM MẪU 10 DÒNG THEO TỪNG NHÓM NGHIỆP VỤ ---
sample_10 = df_merged.sample(10, random_state=42)

# Nhóm 1: Định danh & Truy vết nguồn (7 cột)
group_identity = ['source', 'source_job_id', 'job_url', 'crawled_at', 'raw_date', 'raw_file_name', 'listing_file_name']
print(' NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)')
display(sample_10[group_identity])

# Nhóm 2: Thông tin Công ty (7 cột)
group_company = ['company_name_raw', 'company_url', 'company_logo_url', 'company_size_raw', 'company_location_raw', 'industry_raw', 'location_raw']
print('\n NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)')
display(sample_10[group_company])

# Nhóm 3: Vị trí, Lương & Đãi ngộ tuyển dụng (8 cột)
group_job_specs = ['title_raw', 'salary_raw', 'experience_raw', 'education_raw', 'work_mode_raw', 'employment_type_raw', 'working_time_raw', 'deadline_raw']
print('\n NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)')
display(sample_10[group_job_specs])

# Nhóm 4: Kỹ năng & Ngày tháng (6 cột)
group_skills = ['skills_raw', 'required_skills_raw', 'preferred_skills_raw', 'posted_at_raw', 'published_at_raw', 'updated_at_raw']
print('\n NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)')
display(sample_10[group_skills])

# Nhóm 5: Nội dung chi tiết văn bản (4 cột)
group_texts = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw']
print('\n NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)')
display(sample_10[['title_raw'] + group_texts].map(lambda x: (str(x)[:80] + '...') if len(str(x)) > 80 else str(x)))


 NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)


,source,source_job_id,job_url,crawled_at,raw_date,raw_file_name,listing_file_name
4562,topcv,2344892,https://www.topcv.vn/viec-lam/chuyen-vien-graphic-design...,2026-10-30T10:42:42.136061,2026-10-30,TCV001.json,page_001.json
691,topcv,2190275,https://www.topcv.vn/viec-lam/project-manager/2190275.html,2026-10-11T21:18:17.514708,2026-10-11,TCV010.json,page_010.json
3007,topcv,2298417,https://www.topcv.vn/viec-lam/senior-design-executive/22...,2026-10-17T10:35:24.856234,2026-10-17,TCV001.json,page_001.json
1101,vietnamworks,2114067,https://www.vietnamworks.com/senior-sales-executive-even...,2026-10-11T11:59:23.179617,2026-10-11,VNW004.json,page_004.json
5004,topcv,2302662,https://www.topcv.vn/viec-lam/thuc-tap-sinh-tester-thu-n...,2026-11-03T08:18:21.137213,2026-11-03,TCV001.json,page_001.json
4237,topdev,2140124,https://topdev.vn/viec-lam/nhan-vien-che-ban-webtoon-man...,2026-10-28T16:12:49.042253,2026-10-28,TC001.json,page_001.json
439,topcv,2326532,https://www.topcv.vn/viec-lam/tester-qa/2326532.html,2026-10-11T20:15:05.172517,2026-10-11,TCV008.json,page_007.json
5089,topcv,1514741,https://www.topcv.vn/viec-lam/chuyen-vien-ba/1514741.html,2026-11-03T08:35:41.990385,2026-11-03,TCV001.json,page_001.json
4610,topcv,2268557,https://www.topcv.vn/viec-lam/chuyen-vien-kinh-doanh-pha...,2026-10-30T10:59:42.305307,2026-10-30,TCV001.json,page_001.json
1199,vietnamworks,2111931,https://www.vietnamworks.com/senior-data-governance-exce...,2026-10-11T19:43:22.082885,2026-10-11,VNW005.json,page_005.json



 NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)


,company_name_raw,company_url,company_logo_url,company_size_raw,company_location_raw,industry_raw,location_raw
4562,CÔNG TY TNHH THƯƠNG MẠI VÀ DỊCH VỤ ĐỒ UỐNG VIETBLEND,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-va-...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"Số 10/1, ngõ 55 Huỳnh Thúc Kháng, P. Láng, Hà Nội","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...","Huỳnh Thúc Kháng, Phường Láng, Hà Nội"
691,Công ty TNHH Yopaz,https://www.topcv.vn/cong-ty/cong-ty-tnhh-yopaz/56828.html,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"Tầng 3, Viet Tower, số 1 Thái Hà, Đống Đa, Hà Nội","IT phần mềm, Công nghệ thông tin","Số 1 Thái Hà, Phường Đống Đa, Hà Nội"
3007,CÔNG TY CỔ PHẦN THƯƠNG MẠI KOVA,https://www.topcv.vn/cong-ty/kova-commerce/263712.html,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"92G-92H Nguyễn Hữu Cảnh, Phường Thạnh Mỹ Tây, TP Hồ Chí ...","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...","KOVA Center – 92G-92H Nguyễn Hữu Cảnh, Phường Thạnh Mỹ T..."
1101,Netnam Corp.,https://www.vietnamworks.com/nha-tuyen-dung/netnam-corp-...,https://images.vietnamworks.com/pictureofcompany/00/1112...,100-499,"Floor 12, 07 Nguyen Thi Minh Khai Street, Sai Gon ward, ...",Hệ thống CNTT & Thiết bị (Kinh Doanh > Bán Hàng/Phát Tri...,"VoV Building, 07 Nguyễn Thị Minh Khai, Phường Sài Gòn, H..."
5004,NGÂN HÀNG THƯƠNG MẠI CỔ PHẦN KỸ THƯƠNG VIỆT NAM,https://www.topcv.vn/cong-ty/ngan-hang-thuong-mai-co-pha...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,10000+ nhân viên,"Số 6 Quang Trung, phường Trần Hưng Đạo, Quận Hoàn Kiếm, ...","Kinh doanh / Bán hàng, Ngân hàng / Tài chính, Bảo hiểm","Phường Ba Đình, Hà Nội"
4237,DaouKiwoom Innovation,https://topdev.vn/nha-tuyen-dung/daoukiwoom-innovation-9...,https://assets.saramin.vn/images/2026/03/20/TopDev-rUyqQ...,100-499 Nhân viên,"AP Tower, 518B Äiá»n BiÃªn Phá»§, PhÆ°á»ng 21, Quáº­n...",Finance,Hồ Chí Minh
439,CÔNG TY TNHH SIMPLE SOLUTION,https://www.topcv.vn/cong-ty/cong-ty-tnhh-simple-solutio...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"7/7 Mê Linh, Phường 1, TP Bảo Lộc, Lâm Đồng","IT phần mềm, Công nghệ thông tin","7/7 Mê Linh, Phường 1 Bảo Lộc, Lâm Đồng"
5089,Công ty cổ phần công nghệ Isofh,https://www.topcv.vn/cong-ty/cong-ty-co-phan-cong-nghe-i...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"Tòa VOV, 35 Nguyễn Đình Chiểu, Lê Đại Hành, Hai Bà Trưng...","IT phần mềm, Công nghệ thông tin, IT Phần cứng / Mạng","Tòa nhà VOV, 35 Nguyễn ĐÌnh Chiểu, Phường Hai Bà Trưng, ..."
4610,CÔNG TY TNHH ĐẦU TƯ CÔNG NGHỆ REDSUN,https://www.topcv.vn/cong-ty/cong-ty-tnhh-dau-tu-cong-ng...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"159 Điện Biên Phủ, Phường Gia Định, TP. HCM","Kinh doanh / Bán hàng, IT phần mềm, Công nghệ thông tin","159 Điện Biên Phủ, Phường Gia Định, Hồ Chí Minh"
1199,CÔNG TY TÀI CHÍNH TÍN DỤNG TIÊU DÙNG TRÁCH NHIỆM HỮU HẠN...,https://www.vietnamworks.com/nha-tuyen-dung/cong-ty-tai-...,https://images.vietnamworks.com/pictureofcompany/e9/1147...,5.000-9.999,"Trụ sở chính: Tầng 6, Tòa nhà Gelex, 52 Lê Đại Hành, Hà Nội",Tài Chính (Công Nghệ Thông Tin/Viễn Thông > Data Enginee...,"Tầng 6, 52 phố Lê Đại Hành, phường Lê Đại Hành, Quận Hai..."



 NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)


,title_raw,salary_raw,experience_raw,education_raw,work_mode_raw,employment_type_raw,working_time_raw,deadline_raw
4562,Chuyên Viên Graphic Designer (F&B),13000000 - 16000000,3 năm,Cao Đẳng trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-28T23:59:59+07:00
691,Project Manager,Tới 55 triệu,3 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-04T23:59:59+07:00
3007,Senior Design Executive,Thoả thuận,2 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-15T23:59:59+07:00
1101,Senior Sales Executive (Event/marketing),"$ 600-2,000 /tháng",1 năm,,Onsite,Nhân viên,Toàn thời gian,2026-10-30T23:59:59+07:00
5004,Thực Tập Sinh Tester Thu Nhập Up To 8tr,Thoả thuận,Không yêu cầu,Cao Đẳng trở lên,Onsite,Nhân viên,Toàn thời gian,2026-12-02T23:59:59+07:00
4237,Nhân viên chế bản - Webtoon/ Manga - Full-time,8000000 - 12000000 MONTH,Không yêu cầu,,Full-time,Chuyên viên,,2026-11-21
439,Tester/QA,Thoả thuận,2 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-08T23:59:59+07:00
5089,Chuyên Viên BA,Thoả thuận,1 năm,Cao Đẳng trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-21T23:59:59+07:00
4610,Chuyên Viên Kinh Doanh Phần Mềm - Mô Hình SaaS,12000000 - 20000000,1 năm,Cao Đẳng trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-18T23:59:59+07:00
1199,Senior Data Governance Excecutive,Thương lượng,2 năm,Cử nhân,Onsite,Nhân viên,,2026-10-25T23:59:59+07:00



 NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)


,skills_raw,required_skills_raw,preferred_skills_raw,posted_at_raw,published_at_raw,updated_at_raw
4562,"3 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...","3 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",,2026-10-29,2026-10-29,
691,"3 năm kinh nghiệm chuyên môn, IT Project Manager, IT - P...","3 năm kinh nghiệm chuyên môn, IT Project Manager, IT - P...",,2026-10-05,2026-10-05,
3007,"2 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...","2 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",,2026-10-16,2026-10-16,
1101,"Communication skills, Presentation skills, Aggressive Sa...","Communication skills, Presentation skills, Aggressive Sa...",,2026-09-30T16:25:28+07:00,2026-09-30T16:25:28+07:00,2026-10-08T18:08:36+07:00
5004,"Không yêu cầu kinh nghiệm chuyên môn, Quan hệ khách hàng...","Không yêu cầu kinh nghiệm chuyên môn, Quan hệ khách hàng...",,2026-11-02,2026-11-02,
4237,Photoshop,Photoshop,,2026-10-28,2026-10-28,
439,"2 năm kinh nghiệm chuyên môn, Software Tester (Automatio...","2 năm kinh nghiệm chuyên môn, Software Tester (Automatio...",,2026-10-09,2026-10-09,
5089,"1 năm kinh nghiệm chuyên môn, Chuyên môn Công nghệ thông...","1 năm kinh nghiệm chuyên môn, Chuyên môn Công nghệ thông...",,2026-10-22,2026-10-22,
4610,"1 năm kinh nghiệm chuyên môn, Kinh doanh phần mềm, Direc...","1 năm kinh nghiệm chuyên môn, Kinh doanh phần mềm, Direc...",,2026-10-19,2026-10-19,
1199,"Khoa Học Máy Tính, Quản Trị Dữ Liệu, Bảo Mật Dữ Liệu, Qu...","Khoa Học Máy Tính, Quản Trị Dữ Liệu, Bảo Mật Dữ Liệu, Qu...",,2026-09-25T16:59:48+07:00,2026-09-25T16:59:48+07:00,2026-09-29T09:53:32+07:00



 NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)


,title_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw
4562,Chuyên Viên Graphic Designer (F&B),,Thiết kế các ấn phẩm marketing online và offline: banner...,Tốt nghiệp Cao đẳng/Đại học chuyên ngành Thiết kế đồ họa...,"Mức Lương cạnh tranh, hấp dẫn trên thị trường từ 13Tr-16..."
691,Project Manager,,Quản lý dự án từ kick-off đến bàn giao\nĐiều phối team o...,Có tối thiểu 3 năm kinh nghiệm tại vị trí PM hoặc PM & B...,1. Lương thưởng và các phúc lợi khác\nReview lương 1-2 l...
3007,Senior Design Executive,,1. Chịu trách nhiệm thiết kế\nTriển khai & thực thi phươ...,Tốt nghiệp chuyên ngành thiết kế hoặc các ngành liên qua...,Chế độ bảo hiểm theo quy định của luật.\nChế độ bảo hiểm...
1101,Senior Sales Executive (Event/marketing),NetNam is one of Vietnam’s pioneering Internet Service P...,"Tìm kiếm, tư vấn và phát triển khách hàng sử dụng các dị...","Tốt nghiệp Đại học chuyên ngành Quản trị kinh doanh, Tổ ...","Thưởng: Competitive salary, awards & bonus\nĐào tạo: Var..."
5004,Thực Tập Sinh Tester Thu Nhập Up To 8tr,,A. Thực hiện kiểm thử và đảm bảo chất lượng phần mềm\n- ...,Trình độ và kiến thức chuyên môn\n- Sinh viên năm 3 trở ...,- Được tham gia trực tiếp vào các dự án thực tế cùng đội...
4237,Nhân viên chế bản - Webtoon/ Manga - Full-time,$3e,"Nhận file truyện Webtoon dạng PSD\nNhận file mô tả, hướn...","Độ tuổi: từ 18 tuổi trở lên, làm việc FULL-TIME tại công...","LƯƠNG - THƯỞNG\nLương cạnh tranh, review lương mỗi năm m..."
439,Tester/QA,,"Phân tích yêu cầu, lập kế hoạch và viết test case/checkl...","Có nền tảng CNTT, 2 năm kinh nghiệm Tester/QA; chấp nhận...","Mức lương cạnh tranh, thưởng và phụ cấp theo chính sách ..."
5089,Chuyên Viên BA,,"Phân tích nghiệp vụ với khách hàng:\nTìm hiểu, trau dồi ...",Từ 1 năm kinh nghiệm trở lên với dự án phần mềm\nKinh ng...,Thưởng thâm niên làm việc: 5 triệu/năm\nThưởng tháng lươ...
4610,Chuyên Viên Kinh Doanh Phần Mềm - Mô Hình SaaS,,"Tìm kiếm, tư vấn và chốt hợp đồng khách hàng cho giải ph...",Có từ 1 đến 2 năm kinh nghiệm Sales trong mảng công nghệ...,Thu nhập trung bình: 12–20 triệu/tháng (lương cứng 7–9tr...
1199,Senior Data Governance Excecutive,"SFinance, tiền thân là SHBFinance, là công ty tài chính ...",MỤC TIÊU CÔNG VIỆC\nChuyên viên Quản trị Dữ liệu chịu tr...,TIÊU CHUẨN CHỨC DANH – NĂNG LỰC CẦN THIẾT\nA. NĂNG LỰC K...,"Thưởng: Thu nhập hấp dẫn, cạnh tranh\nChăm sóc sức khoẻ:..."


In [7]:
# ---  BẢNG PHẲNG ĐẦY ĐỦ TOÀN BỘ 39 CỘT (SAMPLE 10 DÒNG) ---
print(' [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):')
display(sample_10)


 [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):


,detail_id,source,source_job_id,job_url,title_raw,company_name_raw,company_url,company_size_raw,company_location_raw,industry_raw,salary_raw,experience_raw,education_raw,location_raw,work_mode_raw,employment_type_raw,working_time_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw,required_skills_raw,preferred_skills_raw,published_at_raw,updated_at_raw,deadline_raw,crawled_at,raw_file_name,raw_date,dq_status,dq_errors,dq_checked_at,_dq_status,_dq_errors,_dq_checked_at,company_logo_url,skills_raw,posted_at_raw,listing_file_name
4562,topcv_2344892,topcv,2344892,https://www.topcv.vn/viec-lam/chuyen-vien-graphic-design...,Chuyên Viên Graphic Designer (F&B),CÔNG TY TNHH THƯƠNG MẠI VÀ DỊCH VỤ ĐỒ UỐNG VIETBLEND,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-va-...,25-99 nhân viên,"Số 10/1, ngõ 55 Huỳnh Thúc Kháng, P. Láng, Hà Nội","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...",13000000 - 16000000,3 năm,Cao Đẳng trở lên,"Huỳnh Thúc Kháng, Phường Láng, Hà Nội",Onsite,Nhân viên,Toàn thời gian,,Thiết kế các ấn phẩm marketing online và offline: banner...,Tốt nghiệp Cao đẳng/Đại học chuyên ngành Thiết kế đồ họa...,"Mức Lương cạnh tranh, hấp dẫn trên thị trường từ 13Tr-16...","3 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",,2026-10-29,,2026-11-28T23:59:59+07:00,2026-10-30T10:42:42.136061,TCV001.json,2026-10-30,valid,[],2026-11-04T21:29:05.612970,valid,[],2026-11-04T21:29:05.612970,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"3 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",2026-10-29,page_001.json
691,topcv_2190275,topcv,2190275,https://www.topcv.vn/viec-lam/project-manager/2190275.html,Project Manager,Công ty TNHH Yopaz,https://www.topcv.vn/cong-ty/cong-ty-tnhh-yopaz/56828.html,100-499 nhân viên,"Tầng 3, Viet Tower, số 1 Thái Hà, Đống Đa, Hà Nội","IT phần mềm, Công nghệ thông tin",Tới 55 triệu,3 năm,Đại Học trở lên,"Số 1 Thái Hà, Phường Đống Đa, Hà Nội",Onsite,Nhân viên,Toàn thời gian,,Quản lý dự án từ kick-off đến bàn giao\nĐiều phối team o...,Có tối thiểu 3 năm kinh nghiệm tại vị trí PM hoặc PM & B...,1. Lương thưởng và các phúc lợi khác\nReview lương 1-2 l...,"3 năm kinh nghiệm chuyên môn, IT Project Manager, IT - P...",,2026-10-05,,2026-11-04T23:59:59+07:00,2026-10-11T21:18:17.514708,TCV010.json,2026-10-11,valid,[],2026-11-04T21:29:05.582998,valid,[],2026-11-04T21:29:05.582998,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"3 năm kinh nghiệm chuyên môn, IT Project Manager, IT - P...",2026-10-05,page_010.json
3007,topcv_2298417,topcv,2298417,https://www.topcv.vn/viec-lam/senior-design-executive/22...,Senior Design Executive,CÔNG TY CỔ PHẦN THƯƠNG MẠI KOVA,https://www.topcv.vn/cong-ty/kova-commerce/263712.html,100-499 nhân viên,"92G-92H Nguyễn Hữu Cảnh, Phường Thạnh Mỹ Tây, TP Hồ Chí ...","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...",Thoả thuận,2 năm,Đại Học trở lên,"KOVA Center – 92G-92H Nguyễn Hữu Cảnh, Phường Thạnh Mỹ T...",Onsite,Nhân viên,Toàn thời gian,,1. Chịu trách nhiệm thiết kế\nTriển khai & thực thi phươ...,Tốt nghiệp chuyên ngành thiết kế hoặc các ngành liên qua...,Chế độ bảo hiểm theo quy định của luật.\nChế độ bảo hiểm...,"2 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",,2026-10-16,,2026-11-15T23:59:59+07:00,2026-10-17T10:35:24.856234,TCV001.json,2026-10-17,valid,[],2026-11-04T21:29:05.596507,valid,[],2026-11-04T21:29:05.596507,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"2 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",2026-10-16,page_001.json
1101,vietnamworks_2114067,vietnamworks,2114067,https://www.vietnamworks.com/senior-sales-executive-even...,Senior Sales Executive (Event/marketing),Netnam Corp.,https://www.vietnamworks.com/nha-tuyen-dung/netnam-corp-...,100-499,"Floor 12, 07 Nguyen Thi Minh Khai Street, Sai Gon ward, ...",Hệ thống CNTT & Thiết bị (Kinh Doanh > Bán Hàng/Phát Tri...,"$ 600-2,000 /tháng",1 năm,,"VoV Building, 07 Nguyễn Thị Minh Khai, Phường Sài Gòn,